## SegFormer Fine-Tunning

Finalmente, se organizó y adaptó y se mejoró la documentación de este cuaderno para refinar el `SegFormerForSemanticSegmentation` de 'HuggingFace' con el **Semantic-Drone Dataset** para eventualmente y como prueba final, entrenarlo también con el **ViVA-DATASET**.

_Nota: Se removieron las decenas de pruebas realizadas tanto para entrenar y desplegar información, como las pruebas de inferencia, despliegue de resultados, experimentos de zonas y demás -- pendiente definir si se organizaran también aquí mismo o se abordarán en otro cuaderno._

In [ ]:
''' Escribir el nombre de la prueba a realizar:
    sera usado para identificar el tensorboard y los modelos que genere '''
experiment_runname = "SDD_Definitivo_1"

In [ ]:
''' Escribir la dirección del Dataset a partir del directorio de ejecución: '''
dataset_dir = "/home/chuyalex/workspace/drone-images-semantic-segmentation/semdrone_processdata_2"
# dataset_dir = "C:/Users/embat/Datasets/ViVa-DATASET"

train2val_ratio = 0.8
# train2val_ratio = 0.9

In [ ]:
''' Escribir la dirección del diccionario de clases del dataset: '''
# dict_dir = '~/chuyalex/SemanticDroneDataset/class_dict.csv'
# labeldict_dir = '/home/chuyalex/workspace/drone-images-semantic-segmentation/semantic_drone_dataset/training_set/gt/semantic/class_dict.csv'
labeldict_dir = "C:/Users/embat/Datasets/SemanticDroneDataset/class_dict.csv"

In [6]:
''' Escribir la dirección del JSON con la configuración del modelo: '''
json_dir = r"C:\Users\embat\Workspace\SegFormer\src\model.json"

In [ ]:
''' En caso de retomar entrenamiento desde uno previamente guardado '''
# weight_fn = None
# pretrained = None
pretrained = "C:/Users/embat/Workspace/SegFormer/best_model_B5.pth"
## Modelo B1 preentrenado
#pretrained = "C:/Users/embat/Workspace/SegFormer/best_model.pth"

### Define PyTorch dataset and dataloaders

Definiendo un [custom PyTorch dataset](https://pytorch.org/tutorials/beginner/data_loading_tutorial.html). Cada elemento del dataset se procesa y crea como un par imágen-anotación, en la que a cada imágen la acompania su mapa de segmentación correspondiente.

In [ ]:
'''System Outils'''
import os
import time
import json
from PIL import Image
#from PIL import ImageFile
#ImageFile.LOAD_TRUNCATED_IMAGES = True

In [ ]:
'''Math Outils'''
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
'''PyTorch'''
from torch.utils.data import Dataset
from torch.utils.tensorboard import SummaryWriter

In [ ]:
'''PyTorch Tensorboard Writer Init'''
writer = SummaryWriter(os.path.join(os.getcwd(),"runs", experiment_runname))

In [ ]:
'''Load Model and Data Configurations'''
## Model Config
with open(json_dir) as f:
    model_json = json.load(f)

## Data Config
color_map = pd.read_csv(labeldict_dir, index_col=False, skipinitialspace=True)
color_map.columns = ["label", "R", "G", "B", "risk"]

label2id = {label: id for id, label in enumerate(color_map.label)}
id2label = {id: label for id, label in enumerate(color_map.label)}
id2color = {id: [R,G,B] for id, (R,G,B) in enumerate(zip(color_map.R, color_map.G, color_map.B))}

print(len(id2label))
print(id2label)
print(id2color)

24
{0: 'unlabeled', 1: 'paved-area', 2: 'dirt', 3: 'grass', 4: 'gravel', 5: 'water', 6: 'rocks', 7: 'pool', 8: 'vegetation', 9: 'roof', 10: 'wall', 11: 'window', 12: 'door', 13: 'fence', 14: 'fence-pole', 15: 'person', 16: 'dog', 17: 'car', 18: 'bicycle', 19: 'tree', 20: 'bald-tree', 21: 'ar-marker', 22: 'obstacle', 23: 'conflicting'}
{0: [0, 0, 0], 1: [128, 64, 128], 2: [130, 76, 0], 3: [0, 102, 0], 4: [112, 103, 87], 5: [28, 42, 168], 6: [48, 41, 30], 7: [0, 50, 89], 8: [107, 142, 35], 9: [70, 70, 70], 10: [102, 102, 156], 11: [254, 228, 12], 12: [254, 148, 12], 13: [190, 153, 153], 14: [153, 153, 153], 15: [255, 22, 96], 16: [102, 51, 0], 17: [9, 143, 150], 18: [119, 11, 32], 19: [51, 51, 0], 20: [190, 250, 190], 21: [112, 150, 146], 22: [2, 135, 115], 23: [255, 0, 0]}


In [18]:
''' RGB Label to Code Label '''
def ssmask_RGB2code(rgb_labels, id2color):

    rgb_labels = np.array(rgb_labels)
    code_labels = np.zeros((rgb_labels.shape[0], rgb_labels.shape[1]), dtype=np.uint8)
    for id, color in id2color.items():
        code_labels[(rgb_labels == color).all(axis=-1)] = id

    return code_labels

In [ ]:
class SemanticSegmentationDataset(Dataset):

    def __init__(self, root_dir, feature_extractor, train=True, proportion=train2val_ratio):
        """
        Args:
            root_dir (string): Root directory of the dataset containing the images + annotations.
            feature_extractor (SegFormerFeatureExtractor): feature extractor to prepare images + segmentation maps.
            train (bool): Whether to load "training" or "validation" images + annotations.
        """
        self.root_dir = root_dir
        self.feature_extractor = feature_extractor
        self.train = train

        self.img_dir = os.path.join(self.root_dir, "img")
        self.ann_dir = os.path.join(self.root_dir, "msk")
        
        # read images
        image_file_names = []
        for root, dirs, files in os.walk(self.img_dir):
            image_file_names.extend(files)
        self.images = sorted(image_file_names)
        
        # read annotations
        annotation_file_names = []
        for root, dirs, files in os.walk(self.ann_dir):
            annotation_file_names.extend(files)
        self.annotations = sorted(annotation_file_names)
        
        # assert images and annotations quantity
        for idx in range(len(self.images)):
            assert self.images[idx].split('.')[-2] == self.annotations[idx].split('.')[-2], "Mask {} don't match Image {}".format(self.annotations[idx], self.images[idx])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        
        image = Image.open(os.path.join(self.img_dir, self.images[idx]))
        annotation = Image.open(os.path.join(self.ann_dir, self.annotations[idx]))
        annotation_2d = ssmask_RGB2code(annotation, id2color)

        # randomly crop + pad both image and segmentation map to same size
        # feature extractor will also reduce labels!
        encoded_inputs = self.feature_extractor(image, Image.fromarray(annotation_2d), return_tensors="pt")

        # remove batch dimension
        for k,v in encoded_inputs.items():
            encoded_inputs[k].squeeze_()

        return encoded_inputs

In [ ]:
'''Herramientas del HuggingFace
    Feature Extractor y DataLoader'''
from transformers import SegformerFeatureExtractor
from torch.utils.data import DataLoader

c:\Users\embat\Workspace\SegFormer\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Nota: Se puede usar el `reduce_labels=True` cuando se desea quitar la etiqueta 0, generalmente siendo el "background" o cuando no se tiene etiqueta 0, porque las etiquetas sí tienen que empezar en 0. Al activarse esta función, las anotaciones se reducen en 1 y las que sean 0 se reemplazan por 255, el cual es el `ignore_index` de la función de pérdida del SegFormer.

In [ ]:
'''Generación de Manejadores del Dataset'''
feature_extractor = SegformerFeatureExtractor(reduce_labels=True, size=(512, 768))

trainset = SemanticSegmentationDataset(root_dir=os.path.join(dataset_dir, "train"), feature_extractor=feature_extractor, train=True, proportion=train2val_ratio)
valset = SemanticSegmentationDataset(root_dir=os.path.join(dataset_dir, "val"), feature_extractor=feature_extractor, train=False)

print("Number of training images:   ", len(trainset))
print("Number of validation images: ", len(valset))

Number of training images:    9000
Number of validation images:  1000


Next, we define corresponding dataloaders.

In [ ]:
'''Generación de Dataloaders'''
trainloader = DataLoader(trainset, batch_size=4, shuffle=True, num_workers=0)
validloader = DataLoader(valset, batch_size=8, num_workers=0)

print(trainloader.num_workers, validloader.num_workers)
trainloader.batch_size, validloader.batch_size

0 0


(2, 4)

### Definiendo el SegFormer

Aquí se carga el modelo y se equipa el encoder con los pesos preentrenados con el ImageNet-1k.

El SegFormer viene en 6 tamaños: desde `nvidia/mit-b0` hasta `nvidia/mit-b5`.
La configuración de cada modelo está en el [hub](https://huggingface.co/models?other=segformer).

También se cargan `id2label` y `label2id` que permiten visualizar los resultados al hacer la inferencia.

In [29]:
from transformers import SegformerForSemanticSegmentation
#from huggingface_hub import cached_download, hf_hub_url

In [ ]:
'''Definición y Configuración del SegFormer'''
model = SegformerForSemanticSegmentation.from_pretrained(#"nvidia/mit-b0",   # 188MiB,   val=1660MiB,    train=2172MiB
                                                         "nvidia/mit-b5",   # 508MiB,   val=3706MiB,    train=9724MiB
                                                         num_labels=24,
                                                         id2label=id2label, 
                                                         label2id=label2id,
)

Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/mit-b5 and are newly initialized: ['decode_head.batch_norm.bias', 'decode_head.batch_norm.num_batches_tracked', 'decode_head.batch_norm.running_mean', 'decode_head.batch_norm.running_var', 'decode_head.batch_norm.weight', 'decode_head.classifier.bias', 'decode_head.classifier.weight', 'decode_head.linear_c.0.proj.bias', 'decode_head.linear_c.0.proj.weight', 'decode_head.linear_c.1.proj.bias', 'decode_head.linear_c.1.proj.weight', 'decode_head.linear_c.2.proj.bias', 'decode_head.linear_c.2.proj.weight', 'decode_head.linear_c.3.proj.bias', 'decode_head.linear_c.3.proj.weight', 'decode_head.linear_fuse.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Ajuste Fino del Modelo

Para adaptar el modelo al tipo de información que va a recibir, se hizo un refinamiento al modelo nativamente en PyTorch, usando el optimizador AdamW usando la misma tasa de aprendizaje que la reportada en el [artículo](https://arxiv.org/abs/2105.15203).

Durante el entrenamiento, también se registraron las métricas; Intersección sobre Unión promedio (mIoU) y precisión balanceada (accuracy).

In [ ]:
import torch
#import torch.nn as nn
from torch import nn

In [ ]:
#from tqdm.notebook import tqdm
#from datasets import load_metric
#from torchmetrics.classification import MulticlassAccuracy
#from sklearn.metrics import accuracy_score
from sklearn.metrics import balanced_accuracy_score

import warnings
# https://stackoverflow.com/questions/879173/how-to-ignore-deprecation-warnings-in-python
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
# define optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=0.00006)

# define loss function
loss_fn = nn.CrossEntropyLoss()

# move model to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

SegformerForSemanticSegmentation(
  (segformer): SegformerModel(
    (encoder): SegformerEncoder(
      (patch_embeddings): ModuleList(
        (0): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(3, 64, kernel_size=(7, 7), stride=(4, 4), padding=(3, 3))
          (layer_norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        )
        (1): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        )
        (2): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(128, 320, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
        )
        (3): SegformerOverlapPatchEmbeddings(
          (proj): Conv2d(320, 512, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
          (layer_norm): LayerNorm((512,), eps=1e-05, elementwise_affine=True)

In [ ]:
'''Continuar Entrenamiento Previo'''
# inicializando variables
epoch = None
running_time = 0
best_loss = float('inf')

# cargando modelo previamente ajustado
if pretrained is not None:
    if os.path.isfile(pretrained):
        print("Loading checkpoint '{}'".format(pretrained))
        checkpoint = torch.load(pretrained, weights_only=False)
        model.load_state_dict(checkpoint['model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        epoch = checkpoint['epoch']
        # best_loss = checkpoint['best_loss']
        running_time = checkpoint['running_time']
        print("Loaded checkpoint '{}' (epoch {})".format(pretrained, checkpoint['epoch']))
    else:
        print("No checkpoint found at '{}'. Will create new checkpoint.".format(pretrained))
else:
    print("Starting new checkpoint.".format(pretrained))
    #pretrained = os.path.join(os.getcwd(), "runs", experiment_runname + "_EOTcheckpoint_{}.pth.tar".format(datetime.now().strftime("%Y%m%d_%H%M%S")))
    pretrained = os.path.join(os.getcwd(), "runs", experiment_runname, "eot_model_at_{}e.pth.tar".format(model_json["epochs"]))

Loading checkpoint 'C:/Users/embat/Workspace/SegFormer/best_model_B5.pth'
Loaded checkpoint 'C:/Users/embat/Workspace/SegFormer/best_model_B5.pth' (epoch 14)


In [ ]:
'''Pruebas de Batch para Aprovechar lo más Posible la GPU '''
# with open(json_dir) as f:
#         model_json = json.load(f)
# #train_dataloader = DataLoader(train_dataset, batch_size=model_json['train_batch_size'], shuffle=True, num_workers=model_json['train_batch_size'])
# train_dataloader = DataLoader(train_dataset, batch_size=model_json['train_batch_size'], shuffle=True)
# #valid_dataloader = DataLoader(valid_dataset, batch_size=model_json['val_batch_size'], num_workers=model_json['val_batch_size'])
# valid_dataloader = DataLoader(valid_dataset, batch_size=model_json['val_batch_size'])
# train_dataloader.batch_size, valid_dataloader.batch_size

""" Ya no sirve mucho este bloque pues cuando explota la RAM del GPU tengo que reiniciar el programa para que el espacio se libere """
# Hay que investigar bien lo de tener limpio ese espacio despues de cada prueba de memoria de GPU

(2, 4)

In [ ]:
'''Estadísticas Iniciales del Modelo antes de Entrenamiento (Epoch 0)'''
sum_valoss = 0.0
sum_valacc = 0.0

t0_e = time.time()

model.eval()
with torch.no_grad():
    for j, data in enumerate(validloader, 1):
        images = data["pixel_values"].to(device)
        gtruth = data["labels"].to(device)

        outputs = model(pixel_values=images, labels=gtruth)
        loss, logits = outputs.loss, outputs.logits

        upsampled_logits = nn.functional.interpolate(logits, size=gtruth.shape[-2:], mode="bilinear", align_corners=False)
        predicted = upsampled_logits.argmax(dim=1)

        accuracy = balanced_accuracy_score(gtruth.cpu().numpy().flatten(), predicted.cpu().numpy().flatten())

            
        if device == 'cuda':
            torch.cuda.empty_cache()

        sum_valoss += loss.item()
        sum_valacc += accuracy

        print('|{}| Batch {} > Loss : {:.4f}, Bal Acc : {:.4f}'.format(int(j*10/len(validloader))*"█"+(10-int(j*10/len(validloader)))*"▒", j, loss.item(), accuracy), end='\r')

print('|{}| Batch {} > Loss : {:.4f}, Bal Acc : {:.4f}'.format(int(j*10/len(validloader))*"█"+(10-int(j*10/len(validloader)))*"▒", j, loss.item(), accuracy))
print('Epoch {} Validation Average > Loss : {:.4f}, Bal Acc : {:.4f}'.format(0, (sum_valoss / j), (sum_valacc / j)))
print("Epoch {} completed in {}.".format(0, time.strftime("%Hh %Mm %Ss", time.gmtime(time.time() - t0_e))), end='\n')
            
writer.add_scalar('Validation_Loss', sum_valoss / j, 0)
writer.add_scalar('Validation_Accuracy', sum_valacc / j, 0)

|██████████| Batch 250 > Loss : 8.6241, Bal Acc : 0.00237
Epoch 0 Validation Average > Loss : 9.2783, Bal Acc : 0.0084
Epoch 0 completed in 01h 28m 23s.


In [ ]:
t0 = time.time()

for i in range(epoch + 1 if epoch is not None else 1, model_json["epochs"] + 1):
    print('Epoch {}:'.format(i))
    
    t0_e = time.time()

    '''========== Training =========='''
    sum_trainloss = 0.0
    sum_trainacc = 0.0

    model.train()
    for j, data in enumerate(trainloader, 1):
        images = data["pixel_values"].to(device)
        gtruth = data["labels"].to(device)

        optimizer.zero_grad()
        outputs = model(pixel_values=images, labels=gtruth)
        
        loss, logits = outputs.loss, outputs.logits
        loss.backward()
        optimizer.step()

        upsampled_logits = nn.functional.interpolate(logits, size=gtruth.shape[-2:], mode="bilinear", align_corners=False)
        predicted = upsampled_logits.argmax(dim=1)

        #accuracy = balanced_accuracy_score(SemanticSegmentationModel.premetricproccess(output), gtruth.cpu().numpy().flatten())
        accuracy = balanced_accuracy_score(gtruth.cpu().numpy().flatten(), predicted.cpu().numpy().flatten())
        
        if device == 'cuda':
            torch.cuda.empty_cache()

        sum_trainloss += loss.item()
        sum_trainacc += accuracy

        print('|{}| Batch {} > Loss : {:.4f}, Bal Acc : {:.4f}'.format(int(j*10/len(trainloader))*"▓"+(10-int(j*10/len(trainloader)))*"░", j, loss.item(), accuracy), end='\r')
        #break

    print('|{}| Batch {} > Loss : {:.4f}, Bal Acc : {:.4f}'.format(int(j*10/len(trainloader))*"▓"+(10-int(j*10/len(trainloader)))*"░", j, loss.item(), accuracy))
    print('Epoch {} Train Average > Loss : {:.4f}, Bal Acc : {:.4f}'.format(i, (sum_trainloss / j), (sum_trainacc / j)), end='\n')
        
    writer.add_scalar('Training_Loss', sum_trainloss / j, i)
    writer.add_scalar('Training_Accuracy', sum_trainacc / j, i)
    

    '''========== Validation =========='''
    sum_valoss = 0.0
    sum_valacc = 0.0

    model.eval()
    with torch.no_grad():
        for j, data in enumerate(validloader, 1):
            images = data["pixel_values"].to(device)
            gtruth = data["labels"].to(device)
            
            outputs = model(pixel_values=images, labels=gtruth)
            
            loss, logits = outputs.loss, outputs.logits
            
            upsampled_logits = nn.functional.interpolate(logits, size=gtruth.shape[-2:], mode="bilinear", align_corners=False)
            predicted = upsampled_logits.argmax(dim=1)

            accuracy = balanced_accuracy_score(gtruth.cpu().numpy().flatten(), predicted.cpu().numpy().flatten())
        
            if device == 'cuda':
                torch.cuda.empty_cache()

            sum_valoss += loss.item()
            sum_valacc += accuracy

            print('|{}| Batch {} > Loss : {:.4f}, Bal Acc : {:.4f}'.format(int(j*10/len(validloader))*"█"+(10-int(j*10/len(validloader)))*"▒", j, loss.item(), accuracy), end='\r')
            #break

    print('|{}| Batch {} > Loss : {:.4f}, Bal Acc : {:.4f}'.format(int(j*10/len(validloader))*"█"+(10-int(j*10/len(validloader)))*"▒", j, loss.item(), accuracy))
    print('Epoch {} Validation Average > Loss : {:.4f}, Bal Acc : {:.4f}'.format(i, (sum_valoss / j), (sum_valacc / j)))
    #print(t0-time.time(), j)
    print("Epoch {} completed in {}.".format(i, time.strftime("%Hh %Mm %Ss", time.gmtime(time.time() - t0_e))), end='\n\n')
    #exit()
        
    writer.add_scalar('Validation_Loss', sum_valoss / j, i)
    writer.add_scalar('Validation_Accuracy', sum_valacc / j, i)
    

    '''---------- Overfitting Validation ----------'''
    if sum_valoss / j < best_loss:
        print("[Metrics improved from {} to {}: updating Best Model state for {}]".format(best_loss, sum_valoss/j, experiment_runname))
        best_loss = sum_valoss / j
        torch.save({
            'epoch': i,
            'running_time': time.time() - t0,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'loss_fn': loss_fn,
            }, os.path.join(os.getcwd(),'runs', experiment_runname, 'best_model.pth'))
    #break

# print("Training complete at {}.".format(time.time() - t0).strftime("%H:%M:%S"))
# https://www.reddit.com/r/sysadmin/comments/afh059/anydesk_keyboard_problem/#DesuThorr
# print(time.strftime("%H:%M:%S", time.gmtime(time.time() - t0)))
# https://stackoverflow.com/questions/60133202/how-to-format-time-time-using-strftime-in-python
# print("Training complete at {}. \nSaving last model.".format(time.strftime("%Hh %Mm %Ss", time.gmtime(time.time() - t0))))
print("Training complete at {}. \nSaving last model.".format(time.strftime("%Hh %Mm %Ss", time.gmtime(running_time + time.time() - t0))))
#save_checkpoint({'epoch': run_epoch, 'state_dict': model.state_dict(), 'optimizer' : optimizer.state_dict()}, weight_fn)
torch.save({'epoch': i, 'running_time': (time.time() - t0), 'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(), 'loss_fn': loss_fn,}, pretrained)

Epoch 1:
|▓▓▓▓▓▓▓▓▓▓| Batch 640 > Loss : 0.6384, Bal Acc : 0.4551
Epoch 1 Train Average > Loss : 0.7238, Bal Acc : 0.5092
|██████████| Batch 80 > Loss : 0.1894, Bal Acc : 0.5873
Epoch 1 Validation Average > Loss : 0.3002, Bal Acc : 0.5646
Epoch 1 completed in 00h 31m 53s.

[Metrics improved from inf to 0.30018670968711375: updating Best Model state for SDD_BIGexperiment_3]
Epoch 2:
|▓▓▓▓▓▓▓▓▓▓| Batch 640 > Loss : 0.4402, Bal Acc : 0.4840
Epoch 2 Train Average > Loss : 0.3534, Bal Acc : 0.6229
|██████████| Batch 80 > Loss : 0.1765, Bal Acc : 0.5586
Epoch 2 Validation Average > Loss : 0.2879, Bal Acc : 0.6053
Epoch 2 completed in 00h 31m 22s.

[Metrics improved from 0.30018670968711375 to 0.28792431866750123: updating Best Model state for SDD_BIGexperiment_3]
Epoch 3:
|▓▓▓▓▓▓▓▓▓▓| Batch 640 > Loss : 0.2737, Bal Acc : 0.7481
Epoch 3 Train Average > Loss : 0.2522, Bal Acc : 0.6622
|██████████| Batch 80 > Loss : 0.1811, Bal Acc : 0.5665
Epoch 3 Validation Average > Loss : 0.2728, Bal Acc : 

NameError: name 'criterion' is not defined